# Solar Active-Region Detection — Kaggle ROLLING run (max memory preset)

**One cell to run** (safe to re-run any time: stops the old run, keeps all
compatible data, resumes).

Right-hand panel (**Session options**):

1. **Internet: ON** — required for the download.
2. **GPU: T4 x2** — both are used automatically (DataParallel).
3. **Persistence: Files only** — so each 12-hour session end KEEPS your data + model.

## How the rolling window works

Kaggle's disk is capped at 20 GiB, so instead of one fixed dataset this run
trains on a **continuously rotating window of the newest ~1,100 frames**:
each download batch retires the oldest frames, and their disk space is
reused for new frames hours later. Every retired frame's **name is recorded
permanently** (frame_manifests_retired/) so no frame is ever re-downloaded —
over days the model sees thousands of unique frames it has never seen.
Validation frames are never retired (they are the scoreboard).

## The "memory" stack (so rotation doesn't cause forgetting)

- **BASE_CHANNELS=64** — 33M-param model (2x the 48-base), the biggest in the
  repo; more capacity = more stored knowledge (batch auto-shrinks to fit VRAM)
- **LR=1e-4** — a third of the default learning rate, so each new update
  overwrites less of the old knowledge (classic anti-interference setting)
- **EMA (built in)** — best.pt is a long-term average of the weights, the
  smoothest "memory" of everything the model has learned
- **3-channel float16 tiles** — ~11 MB/frame, which is what fits 1,100 frames
  under the 20 GiB wall (13-channel frames are 4.3x bigger and cannot)

When a session ends (12 h): run this same cell again — it resumes (data,
retired-name record and checkpoints all persist). The newest model is
mirrored to /kaggle/output (Output tab) every 5 min.

In [ ]:
%%bashset -xexport HOME=/kaggle/workingcd /kaggle/working# 1) Stop any previous run (lock PID + trainer + downloader), and clean orphaned temp files:P=$(cat /kaggle/working/solar_results/arpil/run_forever.lock 2>/dev/null)[ -n "$P" ] && kill -TERM "$P" 2>/dev/nullpkill -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullsleep 5pkill -9 -f "SOALR/scripts/train_streaming.py" 2>/dev/nullpkill -9 -f "SOALR/scripts/build_arpil_resumable.py" 2>/dev/nullrm -rf /tmp/arpil_resume_* 2>/dev/null# 2) One-time data-format check: the rolling preset needs 3-channel FLOAT16 tiles#    (~11 MB/frame). If a previous run left float32 tiles (old code, 2x the size),#    remove them so the 20 GiB budget works. Float16 tiles from a previous#    3-channel run are kept and reused (a head start). The old base-48#    checkpoints are simply ignored by the base-64 trainer (it starts fresh).DTYPE=$(python3 -c "import numpy as np, glob; fs=glob.glob('/kaggle/working/solar_data/arpil/images/*/*.npz'); print(np.load(fs[0])['arr_0'].dtype if fs else 'none')" 2>/dev/null)if [ "$DTYPE" = "float32" ]; then    echo "removing old float32 tiles (2x the disk size of the float16 format)"    rm -rf /kaggle/working/solar_datafi# 3) Fresh code repo (tiny, ~15 s):rm -rf /kaggle/work SOALRgit clone -q -b arena/01a04247-soalr-active-region-detection \    https://github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-.git SOALR \|| { mkdir -p SOALR && wget -qO /tmp/repo.tgz \    https://codeload.github.com/haydenCoder/SOALR-ACTIVE-REGION-DETECTION-MODEL-SUN-/tar.gz/refs/heads/arena/01a04247-soalr-active-region-detection \    && tar xzf /tmp/repo.tgz -C SOALR --strip-components=1; }if [ ! -x SOALR/scripts/run_forever.sh ]; then    echo "STOP: repository download failed. In the notebook Settings, make sure Internet is ON, then re-run this cell."    exit 1ficd SOALR# 4) NO venv on Kaggle (ensurepip is broken there); torch is preinstalled:python3 -m pip install -q -r requirements.txtpython3 -m pip cache purge 2>/dev/nullpython3 -c "import torch; print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available(), '| GPUs:', torch.cuda.device_count())"# 5) Mirror the newest model + log to /kaggle/output every 5 min (downloadable#    from the Output tab after a session ends):mkdir -p /kaggle/output(    while :; do        sleep 300        cp -f /kaggle/working/solar_results/arpil/continuous/best.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/last.pt    /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/continuous/metrics.jsonl /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/run_forever.log      /kaggle/output/ 2>/dev/null        cp -f /kaggle/working/solar_results/arpil/STATUS.md            /kaggle/output/ 2>/dev/null    done) &SAVE_WATCHER=$!# 6) ROLLING MAX-MEMORY preset:#    ROLLING=1 + ROLLING_WINDOW=1100  rotating window of the newest 1,100 frames#                                     (retired names recorded forever; tiles#                                     freed after a 3 h safety gap; val frames#                                     never retired)#    MAX_TOTAL_FRAMES=0               download new frames forever (the window#                                     rotation - not the frame cap - sizes the disk)#    BASE_CHANNELS=64                 33M params (2x the 48-base) = more capacity#    LR=1e-4                          a third of the default LR = less overwrite#                                     of old knowledge (anti-forgetting)#    DEEP_SUPERVISION=1               auxiliary decoder losses#    3 channels x float16 = ~11 MB/frame -> 1,100 frames peak ~18.4 GiB,#    under Kaggle's 20 GiB wall. Both T4s used automatically.CHANNELS="aia171 aia193 hmi_m" \SOLAR_PYTHON="$(command -v python3)" BASE_CHANNELS=64 DEEP_SUPERVISION=1 \ROLLING=1 ROLLING_WINDOW=1100 LR=1e-4 \MIN_FREE_GB=4 MAX_TOTAL_FRAMES=0 FRAMES_PER_CYCLE=200 DOWNLOAD_WORKERS=6 \CPU_HEADROOM=0 TILES_PER_EPOCH=200 VAL_EPOCH=10 VAL_SUBSET=300 \bash scripts/run_forever.shkill "$SAVE_WATCHER" 2>/dev/null

## What you will see

- `torch 2.x.x | CUDA available: True | GPUs: 2` — environment healthy
- Preflight `3 ok`, then `Downloading 200 frames with 6 parallel workers ...`
- `[stream] using 2 GPUs with DataParallel (each batch splits across both)`
- `[stream] epoch=0001 loss=0.7x ...` — finite loss; first `val_dice` at epoch 10
- After the window fills (~1,100 frames): a line like
  `[rolling] 1200 -> 1100 frames on disk (retired 100 this batch; 100 retired total; 0 old tiles freed)`
  — and from then on ~200 NEW frames join the window every batch while the
  oldest retire and their disk is freed 3 h later

## Re-running

This cell is idempotent: it stops any running instance, re-clones the code
(~15 s), and resumes. With **Persistence: Files only**, the data, the retired
name record and the checkpoints all survive — no frame is ever downloaded
twice, across sessions or rotations.